# LangChain 메시지 히스토리 with Gemini

이 노트북은 Google Gemini API를 사용하여 LangChain에서 메시지 히스토리를 관리하는 방법을 보여줍니다.

## 주요 기능:
- 세션별 대화 기록 관리
- Gemini 2.0 Flash 모델 사용
- System Instruction 활용
- 스트리밍 응답 지원

## 1. 환경 설정 및 모델 초기화

In [4]:
import os
from dotenv import load_dotenv
from langchain_core.chat_history import InMemoryChatMessageHistory  # 메모리에 대화 기록을 저장하는 클래스
from langchain_core.runnables.history import RunnableWithMessageHistory  # 메시지 기록을 활용해 실행 가능한 래퍼 클래스
from langchain_core.messages import HumanMessage
from langchain_google_genai import ChatGoogleGenerativeAI  # Google Gemini 모델을 사용하는 랭체인 클래스
from google import genai

In [ ]:
# 환경변수 로드
load_dotenv()
api_key = os.getenv("GOOGLE_API_KEY")
if not api_key:
    raise ValueError("환경변수 GOOGLE_API_KEY가 설정되지 않았습니다.")
llm = ChatGoogleGenerativeAI(
    model="gemini-1.5-pro",
    api_key=api_key,  # 여기서 직접 전달
)

print("✅ 환경변수가 성공적으로 로드되었습니다.")

✅ 환경변수가 성공적으로 로드되었습니다.


In [10]:
# Gemini 모델 초기화 (system_instruction 사용)
model = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    temperature=0.7,
    model_kwargs={
        "system_instruction": (
            "너는 사용자를 도와주는 상담사야. 공감적으로 답하고, "
            "모호하면 짧게 되물어봐. 필요하면 단계별로 안내해줘."
        )
    },
)

print("✅ Gemini 모델이 초기화되었습니다.")

✅ Gemini 모델이 초기화되었습니다.


## 2. 세션별 대화 기록 관리 설정

In [11]:
# 세션별 대화 기록을 저장할 딕셔너리
store = {}

# 세션 ID에 따라 대화 기록을 가져오는 함수
def get_session_history(session_id: str):
    # 만약 해당 세션 ID가 store에 없으면, 새로 생성해 추가함
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()  # 메모리에 대화 기록을 저장하는 객체 생성
    return store[session_id]  # 해당 세션의 대화 기록을 반환

# 모델 실행 시 대화 기록을 함께 전달하는 래퍼 객체 생성
with_message_history = RunnableWithMessageHistory(model, get_session_history)

print("✅ 메시지 히스토리 관리 시스템이 설정되었습니다.")

✅ 메시지 히스토리 관리 시스템이 설정되었습니다.


## 3. 테스트 1: 첫 번째 세션에서 사용자 이름 기억하기

In [12]:
# 세션 abc2 설정
config = {"configurable": {"session_id": "abc2"}}

# 첫 번째 메시지: 자기소개
response = with_message_history.invoke(
    [HumanMessage(content="안녕? 난 이성용이야.")],
    config=config,
)

print("🤖 Gemini 응답:")
print(response.content)

🤖 Gemini 응답:
안녕하세요, 이성용님! 만나서 반갑습니다. 무엇을 도와드릴까요?


## 4. 테스트 2: 같은 세션에서 이름 기억 확인

In [13]:
# 같은 세션에서 이름 질문
response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config,  # 같은 세션 ID 사용
)

print("🤖 Gemini 응답:")
print(response.content)

🤖 Gemini 응답:
당신의 이름은 이성용입니다.


## 5. 테스트 3: 새로운 세션에서는 이름을 모름

In [14]:
# 새로운 세션 abc3 설정
config_new = {"configurable": {"session_id": "abc3"}}

response = with_message_history.invoke(
    [HumanMessage(content="내 이름이 뭐지?")],
    config=config_new,  # 새로운 세션 ID 사용
)

print("🤖 Gemini 응답 (새로운 세션):")
print(response.content)

🤖 Gemini 응답 (새로운 세션):
저는 챗봇이기 때문에 이름이 없습니다. 저는 여러분의 질문에 답하고, 정보를 제공하고, 다양한 작업을 수행하도록 설계된 인공지능입니다. 

혹시 저를 부르고 싶으시다면, 그냥 "챗봇"이라고 불러주시면 됩니다. 😊


## 6. 테스트 4: 첫 번째 세션으로 돌아가서 대화 기록 확인

In [22]:
# 다시 abc2 세션으로 돌아가기
config = {"configurable": {"session_id": "abc2"}}

response = with_message_history.invoke(
    [HumanMessage(content="아까 우리가 무슨 얘기 했지?")],
    config=config,  # 원래 세션 ID 사용
)

print("🤖 Gemini 응답 (원래 세션 복귀):")
print(response.content)

🤖 Gemini 응답 (원래 세션 복귀):
아까는 당신의 이름이 무엇인지에 대해 이야기했어요. 당신이 "안녕? 난 이성용이야."라고 말씀하셨고, 제가 "안녕하세요, 이성용님! 만나서 반갑습니다. 무엇을 도와드릴까요?"라고 답했죠. 그리고 나서 제가 당신의 이름을 다시 물어보셨어요.


## 7. 테스트 5: 스트리밍 응답 테스트

In [23]:
print("🤖 Gemini 스트리밍 응답:")
print("="*50)

# 스트리밍으로 응답 받기
for r in with_message_history.stream(
    [HumanMessage(content="내가 어느 나라 사람인지 추측하고, 그 나라 문화 한 가지를 말해줘")],
    config=config,
):
    # r는 ChatMessage/AIMessage 조각일 수 있음 → content 이어 붙이기
    print(getattr(r, "content", ""), end="", flush=True)

print("\n" + "="*50)

🤖 Gemini 스트리밍 응답:
이름만으로는 어느 나라 사람인지 정확히 추측하기는 어렵습니다. '이성용'이라는 이름은 한국에서 흔히 사용되는 이름이지만, 다른 나라에도 비슷한 발음의 이름이 있을 수 있습니다.

하지만, 한국에서 흔한 이름이므로 **한국인**이라고 추측해 보겠습니다.

한국 문화 중 하나는 **존대말** 문화입니다. 한국어에는 상대방과의 관계, 나이, 사회적 지위 등을 고려하여 사용하는 존대말과 반말이 존재합니다. 특히 처음 만나는 사람이나 나이가 많은 사람에게는 존대말을 사용하는 것이 예의로 여겨집니다. 이는 상호 존중과 배려를 중요하게 생각하는 한국 문화의 특징을 잘 보여줍니다.



## 8. 세션 관리 및 상태 확인

In [24]:
# 현재 생성된 세션들 확인
print("📊 현재 활성 세션들:")
for session_id in store.keys():
    history = store[session_id]
    message_count = len(history.messages)
    print(f"  - 세션 {session_id}: {message_count}개 메시지")

print(f"\n📈 총 {len(store)} 개의 세션이 활성화되어 있습니다.")

📊 현재 활성 세션들:
  - 세션 abc2: 8개 메시지
  - 세션 abc3: 2개 메시지

📈 총 2 개의 세션이 활성화되어 있습니다.


In [ ]:
# 특정 세션의 대화 기록 상세 조회
session_to_check = "abc2"
if session_to_check in store:
    print(f"💬 세션 '{session_to_check}'의 대화 기록:")
    print("-" * 40)
    
    for i, message in enumerate(store[session_to_check].messages, 1):
        speaker = "👤 사용자" if hasattr(message, 'content') and message.__class__.__name__ == "HumanMessage" else "🤖 AI"
        print(f"{i}. {speaker}: {message.content[:100]}{'...' if len(message.content) > 100 else ''}")
else:
    print(f"❌ 세션 '{session_to_check}'를 찾을 수 없습니다.")

## 9. 추가 기능: 새로운 대화 시도

In [ ]:
# 새로운 주제로 대화하기
config = {"configurable": {"session_id": "abc2"}}  # 기존 세션 계속 사용

response = with_message_history.invoke(
    [HumanMessage(content="오늘 날씨가 좋다면 뭘 하면 좋을까?")],
    config=config,
)

print("🤖 Gemini 응답:")
print(response.content)

In [ ]:
# 이전 대화와 연결되는지 확인
response = with_message_history.invoke(
    [HumanMessage(content="아까 내 이름과 함께 추천해줄 수 있어?")],
    config=config,
)

print("🤖 Gemini 응답 (이름 기억 + 맥락 연결):")
print(response.content)

## 10. 정리 및 요약

이 노트북에서 확인한 내용:

✅ **세션별 메시지 히스토리 관리**
- 세션 ID를 통한 대화 기록 분리
- 같은 세션에서는 이전 대화 내용 기억
- 다른 세션에서는 독립적인 대화 시작

✅ **Gemini 2.0 Flash 모델 활용**
- System Instruction으로 AI 성격 설정
- 한국어 대화 지원
- 실시간 스트리밍 응답

✅ **LangChain 통합**
- `RunnableWithMessageHistory`로 히스토리 관리
- `InMemoryChatMessageHistory`로 메모리 저장
- 다양한 메시지 타입 지원

In [ ]:
# 최종 세션 상태 출력
print("🎉 LangChain 메시지 히스토리 with Gemini 테스트 완료!")
print(f"📊 최종 통계:")
print(f"  - 활성 세션 수: {len(store)}")
print(f"  - 총 메시지 수: {sum(len(history.messages) for history in store.values())}")
print("")
print("💡 주요 학습 내용:")
print("  1. Google Gemini API와 LangChain 통합")
print("  2. 세션별 대화 기록 관리")
print("  3. System Instruction 활용")
print("  4. 스트리밍 응답 처리")
print("  5. 메시지 히스토리 상태 모니터링")